In [ ]:
# fr/data-analysis/normal/06-missing-values
# Generated companion notebook for the PyDA course.
# Run cells top-to-bottom (or in any order) to follow the lesson.

print("PyDA — ready 🚀")


In [ ]:
# 💾 Load the course datasets into this environment
# The course data files live in the PyDA repo; pull them so
# `open("…")` / `pd.read_csv("…")` work exactly like on disk.
import os
def _fetch(name, aliases=()):
    if os.path.exists(name):
        return
    url = f"https://raw.githubusercontent.com/abderrahim-lectures/python-data-analysis-course/main/public/datasets/{name}"
    os.system(f"curl -sL -o {name} {url}")
    for alias in aliases:
        if not os.path.exists(alias):
            os.system(f"cp {name} {alias}")

_fetch("titanic.csv", ())


## Pourquoi les valeurs manquantes sont importantes

Presque tous les jeux de données réels contiennent des valeurs manquantes. Si vous les ignorez, les agrégations renvoient NaN, les visualisations échouent et les modèles de machine learning tombent en panne. La première étape de toute analyse consiste à comprendre et traiter les données manquantes.


In [ ]:
import pandas as pd

# titanic.csv ships with the course — load it from the browser file system.
df = pd.read_csv("titanic.csv")


## Détecter les valeurs manquantes

**Vérifier une colonne unique :**


In [ ]:
print(df["Age"].isna().sum())   # 177 missing Age values


**Vérifier toutes les colonnes d'un coup :**


In [ ]:
print(df.isna().sum())


Sortie :


In [ ]:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


**Voir le pourcentage manquant :**


In [ ]:
print((df.isna().sum() / len(df) * 100).round(1))


Sortie :


In [ ]:
Cabin          77.1%
Age            19.9%
Embarked        0.2%
...


Cabin est manquant à 77 % — trop pour être comblé de manière pertinente. Age est manquant à 20 % — cela vaut la peine de tenter de le combler. Embarked n'a que 2 valeurs manquantes — facile à gérer.

## Supprimer les valeurs manquantes

**Supprimer les lignes avec des valeurs manquantes :**


In [ ]:
df_clean = df.dropna()
print(df_clean.shape)   # (183, 12) — lost most rows


C'est trop agressif pour la plupart des jeux de données. Vous perdez 708 lignes sur 891.

**Supprimer les lignes où toutes les valeurs sont manquantes :**


In [ ]:
df_clean = df.dropna(how="all")


**Supprimer les lignes manquant de valeurs dans des colonnes spécifiques :**


In [ ]:
df_clean = df.dropna(subset=["Age", "Embarked"])
print(df_clean.shape)   # (712, 12) — much better


**Supprimer les colonnes avec trop de valeurs manquantes :**


In [ ]:
# Drop columns where more than 50% is missing
threshold = len(df) * 0.5
df_clean = df.dropna(thresh=threshold, axis=1)


## Combler les valeurs manquantes

**Combler avec une constante :**


In [ ]:
df["Embarked"] = df["Embarked"].fillna("S")   # most common port


**Combler avec une statistique :**


In [ ]:
df["Age"] = df["Age"].fillna(df["Age"].median())


**Comblement vers l'avant ou vers l'arrière** — utile pour les séries temporelles :


In [ ]:
# Use the previous valid value to fill gaps
df["Price"] = df["Price"].ffill()

# Use the next valid value
df["Price"] = df["Price"].bfill()


**Combler avec des valeurs différentes par colonne :**


In [ ]:
fill_values = {"Age": df["Age"].median(), "Embarked": "S", "Cabin": "Unknown"}
df = df.fillna(fill_values)


## Choisir une stratégie

| Scénario | Stratégie |
|---|---|
| Valeurs manquantes aléatoires et peu nombreuses (< 5 %) | Supprimer avec `dropna(subset=[...])` |
| Valeurs manquantes dans une colonne numérique | Combler avec la médiane (robuste aux valeurs aberrantes) |
| Valeurs manquantes dans une colonne catégorielle | Combler avec le mode ou « Unknown » |
| Colonne manquante à plus de 50 % | Supprimer toute la colonne |
| Données de séries temporelles | Utiliser `ffill()` ou `bfill()` |

## Pièges courants

**Combler avant de diviser entraînement/test** — cela fuit des informations. Calculez les valeurs de comblement uniquement sur les données d'entraînement, puis appliquez-les aux deux ensembles.

**Supprimer trop agressivement** — vérifiez toujours combien de lignes vous perdez. `dropna()` sans arguments supprime souvent beaucoup plus que prévu.

**Oublier de vérifier** — exécutez toujours `df.isna().sum()` après le comblement pour confirmer qu'aucune valeur NaN ne subsiste.

## Essayez-le

À partir du jeu de données Titanic :
1. Calculez le pourcentage de valeurs manquantes pour chaque colonne
2. Supprimez la colonne Cabin (trop de valeurs manquantes)
3. Comblez Age avec l'âge médian
4. Comblez Embarked avec la valeur la plus courante
5. Vérifiez qu'il ne reste aucune valeur manquante


In [ ]:
import pandas as pd

# titanic.csv ships with the course — load it from the browser file system.
df = pd.read_csv("titanic.csv")

print((df.isna().sum() / len(df) * 100).round(1))

df = df.drop(columns=["Cabin"])
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

print(df.isna().sum())


## Points clés à retenir

- Inspectez toujours les valeurs manquantes d'abord avec `isna().sum()` avant de décider d'une stratégie
- `dropna()` est puissant mais souvent trop agressif sans `subset` ou `thresh`
- `fillna()` avec la médiane ou le mode est la stratégie de comblement la plus courante
- Les colonnes manquantes à plus de 50 % sont généralement mieux supprimées que comblées

## Défi pratique

Chargez le jeu de données Titanic et créez une version nettoyée : supprimez Cabin, comblez Age avec la médiane, comblez Embarked avec le mode. Comparez ensuite le taux de survie avant et après nettoyage. Le nettoyage a-t-il modifié le taux de survie global ? Pourquoi ou pourquoi pas ?


In [ ]:
# The end. Practice on your own — each cell is a minimal, runnable chunk.
